In [0]:
# Instala a biblioteca do kagglehub no ambiente do cluster
%pip install kagglehub

import kagglehub
import os
import pandas as pd

In [0]:
# 1. Cria caixas de input (widgets) no topo do notebook para você alterar facilmente
dbutils.widgets.text(
    "kaggle_user", "", "Seu Usuário do Kaggle"
)
dbutils.widgets.text(
    "dataset_path", "", "Caminho do Dataset (Kaggle)"
)

In [0]:
# 2. Pega os valores que você digitou nos widgets acima
usuario_kaggle = dbutils.widgets.get("kaggle_user").strip()
caminho_dataset = dbutils.widgets.get("dataset_path").strip()

# 3. Puxa a chave do Kaggle do cofre seguro criado anteriormente
token_kaggle = dbutils.secrets.get(scope="segredos", key="kaggle_key")

# 4. Configura as variáveis de ambiente para o Kaggle autenticar
os.environ["KAGGLE_USERNAME"] = usuario_kaggle
os.environ["KAGGLE_KEY"] = token_kaggle

# 5. Faz o download do dataset usando o caminho dinâmico do widget
print(f"A descarregar o dataset '{caminho_dataset}'...")
path_dataset = kagglehub.dataset_download(caminho_dataset)

# 6. Teste de Confirmação: Verifica se a pasta existe e lista os arquivos descarregados
if os.path.exists(path_dataset):
  print(f"\n[SUCESSO] Os arquivos foram descarregados para: {path_dataset}")
  print("\nFicheiros encontrados na pasta:")
  for ficheiro in os.listdir(path_dataset):
    print(f" - {ficheiro}")
else:
  print(
      "\n[ERRO] O diretório de download não foi encontrado. Verifique o caminho"
      " do dataset."
  )

In [0]:
import os
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
ficheiros = os.listdir(path_dataset)

for ficheiro in ficheiros:
  if ficheiro.endswith(".csv"):
    caminho_completo = f"{path_dataset}/{ficheiro}"
    nome_limpo = (
        ficheiro.replace(".csv", "").lower().replace("-", "_").replace(".", "_")
    )
    tabela_destino = f"bronze.{nome_limpo}"

    try:
      # Leitura pelo Pandas no Python local (contorna o bloqueio do Spark UC)
      pdf = pd.read_csv(caminho_completo, low_memory=False)

      # Converte para Spark DataFrame e injeta metadados de governança
      df_bronze = (
          spark.createDataFrame(pdf)
          .withColumn("_ingested_at", current_timestamp())
          .withColumn("_source_file", lit(ficheiro))
      )

      # Salva no formato Delta Lake no schema bronze
      df_bronze.write.format("delta").mode("overwrite").saveAsTable(
          tabela_destino
      )

      print(f"[SUCESSO] Tabela criada: '{tabela_destino}'")

    except Exception as e:
      print(f"[ERRO] Falha ao processar o ficheiro {ficheiro}: {e}")

In [0]:
%sql
SELECT * FROM bronze.transfers
WHERE from_club_name = "Grêmio"
ORDER BY transfer_date DESC
LIMIT 10

In [0]:
import pandas

pandas.read